In [ ]:
# 🚀 LightLLM - Kaggle 30-Hour GPU Training Notebook (Dual T4 High-Speed Edition)
# Step 1: Verify High-End GPU Hardware & VRAM Capacity
!nvidia-smi

In [ ]:
# Step 2: Clone repository & install dependencies
!git clone https://github.com/RABNEER/LightLLM.git
%cd LightLLM
!pip install torch numpy tiktoken tqdm

In [ ]:
# Step 3: Prepare expanded dataset (Facts, Math, Instructions & Greetings)
!python prepare_data.py

In [ ]:
# Step 4: Configure train.py for Stable Dual T4 GPU Training (batch_size=32, ~7GB VRAM, 50,000 steps)
with open('train.py', 'r') as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if line.startswith('batch_size ='):
        new_lines.append('batch_size = 32  # Optimal batch size for Dual T4 GPUs (~7GB VRAM)\n')
    elif line.startswith('max_iters ='):
        new_lines.append('max_iters = 50000  # 50,000 steps deep training\n')
    elif line.startswith('lr_decay_iters ='):
        new_lines.append('lr_decay_iters = 50000\n')
    else:
        new_lines.append(line)

with open('train.py', 'w') as f:
    f.writelines(new_lines)

print('[SUCCESS] train.py configured for Dual T4 GPUs (batch_size=32) & 50,000 steps!')

In [ ]:
# Step 5: Launch High-Speed Cloud GPU Training (Dual T4 Multi-GPU Running!)
!python train.py

In [ ]:
# Step 6: Interactive Chat Test
import torch
from lightllm.model import LightLLM
from lightllm.config import LightLLMConfig
from lightllm.tokenizer import Tokenizer

config = LightLLMConfig()
model = LightLLM(config)
tokenizer = Tokenizer()
checkpoint = torch.load('out/checkpoint.pt', map_location='cuda')
model.load_state_dict(checkpoint['model'], strict=False)
model.to('cuda').eval()

def chat(prompt):
    formatted = f"User: {prompt}\nAssistant:"
    ids = torch.tensor([tokenizer.encode(formatted)], dtype=torch.long).to('cuda')
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=60, temperature=0.2, top_k=5)
    return tokenizer.decode(out[0].tolist()).split('<|endoftext|>')[0]

print("Q: hello ->", chat("hello"))
print("Q: 2+2 ->", chat("2+2"))
print("Q: 7*8 ->", chat("7*8"))
print("Q: what is an apple ->", chat("what is an apple"))
print("Q: what is python ->", chat("what is python"))
print("Q: who created you ->", chat("who created you"))

In [ ]:
# Step 7: Save Checkpoint to Kaggle Output Directory
!cp out/checkpoint.pt /kaggle/working/LightLLM_124M_trained.pt
print('[SUCCESS] Trained weights saved to /kaggle/working/LightLLM_124M_trained.pt (Ready to download!)')